# M2-B1 — Audit de qualité et effet du nettoyage

Les **outils** sont fournis (rapport de qualité, comparaison des modèles). Les **constats**, les **décisions**
et leur **interprétation** sont à toi : réponds dans les cellules ✏️.

In [2]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")
engine = create_engine(os.environ["DATABASE_URL"])

df = pd.read_sql(text("""
    SELECT h.code AS hotel, r.*
    FROM reservations AS r JOIN hotels AS h ON h.hotel_id = r.hotel_id
    ORDER BY r.arrival_date
"""), engine, parse_dates=["arrival_date", "reservation_status_date"])
df.shape

(78703, 35)

## Tâche 2 — L'audit

In [3]:
# Bornes « raisonnables » par colonne numérique : hypothèses de départ, à discuter
BORNES = {
    "lead_time": (0, 730),
    "adults": (1, 10),
    "children": (0, 10),
    "babies": (0, 5),
    "stays_in_weekend_nights": (0, 20),
    "stays_in_week_nights": (0, 50),
    "adr": (0, 1000),
}


def rapport_qualite(d: pd.DataFrame) -> pd.DataFrame:
    """Manquants et valeurs hors bornes, colonne par colonne."""
    lignes = []
    for col in d.columns:
        info = {"colonne": col, "type": str(d[col].dtype),
                "manquants": int(d[col].isna().sum()), "valeurs_distinctes": d[col].nunique()}
        if col in BORNES:
            bas, haut = BORNES[col]
            info["hors_bornes"] = int(((d[col] < bas) | (d[col] > haut)).sum())
        lignes.append(info)
    return pd.DataFrame(lignes).set_index("colonne")


rapport = rapport_qualite(df)
rapport[(rapport["manquants"] > 0) | (rapport.get("hors_bornes", 0) > 0)]

,type,manquants,valeurs_distinctes,hors_bornes
colonne,,,,
lead_time,int64,0,439,1.0
adults,int64,0,14,285.0
children,float64,4,5,0.0
babies,int64,0,5,2.0
country,object,405,159,NaN
agent,float64,10676,284,NaN
company,float64,74216,298,NaN
adr,float64,0,6985,1.0


In [4]:
# Doublons : réservations identiques sur toutes les colonnes, sauf l'identifiant
doublons = df.drop(columns="reservation_id").duplicated(keep="first")
print(doublons.sum(), "lignes en double")

22999 lignes en double


In [5]:
# Exemples : quelles valeurs sont hors bornes, colonne par colonne
for col, (bas, haut) in BORNES.items():
    hors = df[(df[col] < bas) | (df[col] > haut)]
    if len(hors):
        print(f"{col} : {len(hors)} lignes hors [{bas}, {haut}], valeurs {hors[col].value_counts().to_dict()}")

lead_time : 1 lignes hors [0, 730], valeurs {737: 1}
adults : 285 lignes hors [1, 10], valeurs {0: 273, 26: 5, 27: 2, 20: 2, 40: 1, 50: 1, 55: 1}
babies : 2 lignes hors [0, 5], valeurs {9: 1, 10: 1}
adr : 1 lignes hors [0, 1000], valeurs {5400.0: 1}


In [6]:
# Exemples : les lignes extrêmes et les children manquants
COLS_EXEMPLE = ["hotel", "arrival_date", "market_segment", "deposit_type", "adults", "children", "babies",
                "stays_in_weekend_nights", "stays_in_week_nights", "adr", "is_canceled"]
display(df.loc[(df["adr"] > 1000) | (df["babies"] > 5) | (df["lead_time"] > 730), COLS_EXEMPLE])
display(df.loc[df["children"].isna(), COLS_EXEMPLE])

,hotel,arrival_date,market_segment,deposit_type,adults,children,babies,stays_in_weekend_nights,stays_in_week_nights,adr,is_canceled
80,Resort Hotel,2015-07-01,Direct,No Deposit,2,0.0,0,0,0,0.00,0
13759,City Hotel,2015-10-11,Corporate,No Deposit,1,0.0,9,2,1,95.00,0
22709,City Hotel,2016-01-12,Online TA,No Deposit,2,0.0,10,0,2,84.45,0
31828,City Hotel,2016-03-25,Offline TA/TO,Non Refund,2,0.0,0,0,1,5400.00,1


,hotel,arrival_date,market_segment,deposit_type,adults,children,babies,stays_in_weekend_nights,stays_in_week_nights,adr,is_canceled
2940,City Hotel,2015-08-03,Undefined,No Deposit,2,NaN,0,1,0,12.0,1
3189,City Hotel,2015-08-05,Direct,No Deposit,2,NaN,0,0,2,12.0,1
3201,City Hotel,2015-08-05,Undefined,No Deposit,3,NaN,0,0,2,18.0,1
4304,City Hotel,2015-08-13,Online TA,No Deposit,2,NaN,0,2,5,76.5,1


In [7]:
# Manquants de country et agent : le manque est-il aléatoire ? On compare le taux d'annulation
for col in ["country", "agent"]:
    print(df.groupby(df[col].isna())["is_canceled"].agg(["size", "mean"]).round(3)
          .rename(index={False: f"{col} renseigné", True: f"{col} manquant"}), "\n")

                    size   mean
country                        
country renseigné  78298  0.363
country manquant     405  0.138 

                  size   mean
agent                        
agent renseigné  68027  0.386
agent manquant   10676  0.206 



In [8]:
# Doublons : combien de groupes, de quelle taille, de quel segment, et avec quel taux d'annulation
colonnes = list(df.columns.drop("reservation_id"))
en_double = df.duplicated(subset=colonnes, keep=False)   # toutes les lignes d'un groupe, y compris la première
taille_groupes = df[en_double].groupby(colonnes, dropna=False).size()
print(len(taille_groupes), "groupes de lignes identiques ; taille :", taille_groupes.describe().round(1).to_dict())
print("Taux d'annulation :", df.groupby(en_double)["is_canceled"].mean().round(3).to_dict(), "(False = lignes uniques)")
print("Dépôt des lignes en trop :", df.loc[doublons, "deposit_type"].value_counts().to_dict())

pd.DataFrame({
    "part dans les doublons": df.loc[doublons, "market_segment"].value_counts(normalize=True),
    "part dans la base": df["market_segment"].value_counts(normalize=True),
}).round(3)

5458 groupes de lignes identiques ; taille : {'count': 5458.0, 'mean': 5.2, 'std': 8.6, 'min': 2.0, '25%': 2.0, '50%': 2.0, '75%': 4.0, 'max': 180.0}
Taux d'annulation : {False: 0.236, True: 0.585} (False = lignes uniques)
Dépôt des lignes en trop : {'No Deposit': 13188, 'Non Refund': 9760, 'Refundable': 51}


,part dans les doublons,part dans la base
market_segment,,
Aviation,0.000,0.002
Complementary,0.001,0.007
Corporate,0.033,0.047
Direct,0.022,0.101
Groups,0.460,0.177
Offline TA/TO,0.363,0.236
Online TA,0.121,0.430
Undefined,NaN,0.000


### ✏️ Ce que le rapport montre

Base : **78 703 réservations**, juillet 2015 → décembre 2016.

**Valeurs manquantes**

- `children` : **4 lignes**. Toutes au City Hotel, en août 2015, toutes annulées, deux au segment `Undefined` avec un prix de 12 à 18 €. Elles ressemblent à des saisies incomplètes. L'imputation à 0 (règle d'exemple) ne change presque rien.
- `country` : **405 lignes** (0,5 %). Le manque **n'est pas aléatoire** : ces réservations sont annulées à 13,8 % contre 36,3 % pour les autres (le pays est sans doute complété à l'arrivée du client). Le pipeline du modèle les range déjà dans une catégorie `Unknown`, ce qui garde cette information.
- `agent` (10 676) et `company` (74 216) : ici, « manquant » veut dire « pas d'agence » ou « pas d'entreprise ». Ce n'est pas une erreur, et ces colonnes ne sont pas utilisées par le modèle.

**Valeurs hors bornes** (avec les bornes de départ du notebook)

- `adults = 0` : **273 lignes**, détaillées plus bas dans les incohérences métier.
- `adults` de 20 à 55 : **12 lignes**. Toutes au Resort, en septembre-octobre 2015, toutes annulées, toutes à prix nul. Ce ne sont pas de vrais séjours : ça ressemble à des blocs de chambres posés pour un groupe sur une seule ligne, puis libérés.
- `babies` = 9 et 10 : **2 lignes**. 1 ou 2 adultes, séjour normal, honoré : faute de frappe quasi certaine.
- `adr = 5 400 €` : **1 ligne**. City Hotel, 1 nuit, 2 adultes, annulée. Le prix suivant le plus élevé de la base est 508 € : sans doute 54,00 € mal saisi.
- `lead_time = 737` : **1 ligne**. Elle dépasse à peine la borne de 730, qui est arbitraire : réserver 2 ans à l'avance n'est pas impossible. En revanche, cette même ligne a 0 nuit et un prix nul : elle relève d'une autre anomalie.

**Doublons** : **22 999 lignes en trop** (29 % de la base), identiques à une autre sur toutes les colonnes sauf l'identifiant. Elles forment 5 458 groupes de lignes identiques (2 lignes en médiane, jusqu'à 180).

Les doublons ne sont **pas forcément des erreurs** :

- la base n'a **ni identifiant client, ni numéro de chambre**. Deux familles qui réservent la même chambre le même jour, par la même agence, au même tarif, donnent deux lignes identiques ;
- 46 % des doublons viennent du segment `Groups` (18 % de la base) et 36 % de `Offline TA/TO` (24 % de la base). C'est la signature des **réservations de groupe** : une ligne par chambre.

Mais ils pèsent lourd sur le modèle : ils sont annulés à **58 %** contre 24 % pour les lignes uniques, et **93 % des réservations `Non Refund` sont des doublons**. Les supprimer n'est donc pas un nettoyage neutre : ça retire 29 % des lignes et presque tous les acomptes non remboursables. C'est une question pour le client (voir la tâche 3).

### ✏️ Ce que le rapport ne voit pas

Le rapport ne connaît que des bornes colonne par colonne. Cherche les **incohérences métier** : des combinaisons
de valeurs impossibles ou suspectes pour un hôtel. Au moins deux, chiffrées. Écris le code dans les cellules
ci-dessous.



In [9]:
# Variables utiles pour toute l'investigation
nuits = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
personnes = df["adults"] + df["children"].fillna(0) + df["babies"]

# 1. Séjours sans aucune nuit
zero_nuit = nuits == 0
print(f"{zero_nuit.sum()} réservations sans aucune nuit, dont {(df.loc[zero_nuit, 'adr'] == 0).sum()} à prix nul")
print("Statut :", df.loc[zero_nuit, "reservation_status"].value_counts().to_dict())
df.loc[zero_nuit, COLS_EXEMPLE].head()

571 réservations sans aucune nuit, dont 571 à prix nul
Statut : {'Check-Out': 547, 'No-Show': 12, 'Canceled': 12}


,hotel,arrival_date,market_segment,deposit_type,adults,children,babies,stays_in_weekend_nights,stays_in_week_nights,adr,is_canceled
79,Resort Hotel,2015-07-01,Direct,No Deposit,2,0.0,0,0,0,0.0,0
80,Resort Hotel,2015-07-01,Direct,No Deposit,2,0.0,0,0,0,0.0,0
449,Resort Hotel,2015-07-06,Online TA,No Deposit,2,0.0,0,0,0,0.0,0
450,Resort Hotel,2015-07-06,Direct,No Deposit,1,0.0,0,0,0,0.0,0
509,Resort Hotel,2015-07-07,Direct,No Deposit,2,0.0,0,0,0,0.0,0


In [10]:
# 2. Réservations sans adulte
sans_adulte = df["adults"] == 0
print(f"{sans_adulte.sum()} réservations sans adulte : "
      f"{(sans_adulte & (personnes == 0)).sum()} sans personne du tout, "
      f"{(sans_adulte & (personnes > 0)).sum()} avec seulement des enfants ou des bébés")
print("Segment :", df.loc[sans_adulte, "market_segment"].value_counts().to_dict())
df.loc[sans_adulte, COLS_EXEMPLE].sample(5, random_state=0)

273 réservations sans adulte : 113 sans personne du tout, 160 avec seulement des enfants ou des bébés
Segment : {'Online TA': 172, 'Direct': 37, 'Offline TA/TO': 31, 'Groups': 12, 'Complementary': 12, 'Corporate': 9}


,hotel,arrival_date,market_segment,deposit_type,adults,children,babies,stays_in_weekend_nights,stays_in_week_nights,adr,is_canceled
70313,City Hotel,2016-10-31,Online TA,No Deposit,0,2.0,0,1,3,83.50,0
76929,City Hotel,2016-12-18,Online TA,No Deposit,0,0.0,0,0,0,0.00,0
73117,City Hotel,2016-11-17,Corporate,No Deposit,0,0.0,0,4,11,0.00,1
44402,City Hotel,2016-06-03,Groups,No Deposit,0,0.0,0,3,7,0.00,0
58070,City Hotel,2016-08-27,Online TA,No Deposit,0,2.0,0,1,1,86.63,1


In [11]:
# 3. Prix nul : lesquels ont une explication ?
prix_nul = df["adr"] == 0
offert = df["market_segment"] == "Complementary"
prix_nul_inexplique = prix_nul & ~zero_nuit & ~offert
print(pd.Series({
    "prix nul, total": prix_nul.sum(),
    "dont séjour offert (Complementary)": (prix_nul & offert).sum(),
    "dont séjour sans nuit (non offert)": (prix_nul & zero_nuit & ~offert).sum(),
    "dont inexpliqué (nuits > 0, non offert)": prix_nul_inexplique.sum(),
}).to_string())
print("Segment des inexpliqués :", df.loc[prix_nul_inexplique, "market_segment"].value_counts().to_dict())

prix nul, total                            1504
dont séjour offert (Complementary)          479
dont séjour sans nuit (non offert)          554
dont inexpliqué (nuits > 0, non offert)     471
Segment des inexpliqués : {'Groups': 186, 'Offline TA/TO': 162, 'Direct': 68, 'Online TA': 35, 'Corporate': 20}


In [12]:
# 4. Acompte non remboursable : taux d'annulation et profil
display(df.groupby("deposit_type")["is_canceled"].agg(["size", "mean"]).round(3))

nr = df[df["deposit_type"] == "Non Refund"]
print(f"Non Refund non annulées : {(nr['is_canceled'] == 0).sum()} sur {len(nr)}")
print(f"Clients portugais : {(nr['country'] == 'PRT').mean():.1%} | City Hotel : {(nr['hotel'] == 'City Hotel').mean():.1%}"
      f" | lead_time médian : {nr['lead_time'].median():.0f} j (base : {df['lead_time'].median():.0f} j)"
      f" | part de doublons : {doublons[nr.index].mean():.1%}")
pd.DataFrame({
    "Non Refund": nr["market_segment"].value_counts(normalize=True),
    "toute la base": df["market_segment"].value_counts(normalize=True),
}).round(3).fillna(0)

,size,mean
deposit_type,,
No Deposit,68107,0.266
Non Refund,10461,0.992
Refundable,135,0.074


Non Refund non annulées : 85 sur 10461
Clients portugais : 96.2% | City Hotel : 86.6% | lead_time médian : 238 j (base : 65 j) | part de doublons : 93.3%


,Non Refund,toute la base
market_segment,,
Aviation,0.000,0.002
Complementary,0.000,0.007
Corporate,0.014,0.047
Direct,0.001,0.101
Groups,0.611,0.177
Offline TA/TO,0.369,0.236
Online TA,0.005,0.430
Undefined,0.000,0.000


In [13]:
# 5. Groupes de plus de 10 adultes sur une seule ligne
display(df.loc[df["adults"] > 10, COLS_EXEMPLE + ["reservation_status"]])

# 6. Départ (Check-Out) daté autrement que « arrivée + nombre de nuits »
depart = df[df["reservation_status"] == "Check-Out"]
ecart_jours = (depart["reservation_status_date"] - depart["arrival_date"]).dt.days - nuits[depart.index]
print((ecart_jours != 0).sum(), "départs dont la date ne correspond pas à arrivée + nuits")
depart.loc[ecart_jours != 0, ["arrival_date", "reservation_status_date", "stays_in_weekend_nights",
                              "stays_in_week_nights", "adr"]].head()

,hotel,arrival_date,market_segment,deposit_type,adults,children,babies,stays_in_weekend_nights,stays_in_week_nights,adr,is_canceled,reservation_status
6989,Resort Hotel,2015-09-03,Direct,No Deposit,40,0.0,0,0,3,0.0,1,Canceled
7435,Resort Hotel,2015-09-05,Offline TA/TO,No Deposit,26,0.0,0,2,5,0.0,1,Canceled
7716,Resort Hotel,2015-09-07,Direct,No Deposit,50,0.0,0,1,2,0.0,1,Canceled
8554,Resort Hotel,2015-09-12,Offline TA/TO,No Deposit,26,0.0,0,2,5,0.0,1,Canceled
9921,Resort Hotel,2015-09-19,Offline TA/TO,No Deposit,26,0.0,0,2,5,0.0,1,Canceled
10286,Resort Hotel,2015-09-21,Direct,No Deposit,27,0.0,0,1,3,0.0,1,Canceled
10769,Resort Hotel,2015-09-24,Direct,No Deposit,27,0.0,0,1,3,0.0,1,Canceled
11224,Resort Hotel,2015-09-26,Offline TA/TO,No Deposit,26,0.0,0,2,5,0.0,1,Canceled
12261,Resort Hotel,2015-10-03,Offline TA/TO,No Deposit,26,0.0,0,2,5,0.0,1,Canceled
12428,Resort Hotel,2015-10-04,Direct,No Deposit,55,0.0,0,2,0,0.0,1,Canceled


23 départs dont la date ne correspond pas à arrivée + nuits


,arrival_date,reservation_status_date,stays_in_weekend_nights,stays_in_week_nights,adr
103,2015-07-01,2015-07-01,2,5,94.95
868,2015-07-12,2015-07-12,2,5,117.22
1594,2015-07-20,2015-07-20,1,3,104.00
2668,2015-07-30,2015-07-30,0,3,280.74
7730,2015-09-07,2015-09-07,3,7,83.50


### ✏️ Synthèse des incohérences métier

| # | Incohérence | Lignes | À quoi elles ressemblent |
|---|---|---|---|
| 1 | **Séjour sans aucune nuit** (`stays_*` = 0) | **571** | Toutes à prix nul, 96 % honorées (`Check-Out`). Peut-être des chambres louées à la journée (*day use*), peut-être une saisie incomplète. |
| 2 | **Réservation sans adulte** | **273** | 113 sans **personne du tout** (réservation vide) et 160 avec seulement des enfants ou des bébés. Un mineur ne réserve pas seul un hôtel ; 172 de ces lignes viennent d'agences en ligne (erreur de formulaire ?). |
| 3 | **Prix nul sans explication** | **471** | Sur 1 504 prix nuls : 479 sont des séjours offerts (`Complementary`, c'est normal) et 554 des séjours sans nuit (cf. n° 1). Restent 471 séjours avec des nuits, non offerts, surtout `Groups` et `Offline TA/TO` : tarif facturé ailleurs (contrat, note du groupe) ou oubli de saisie ? |
| 4 | **Acompte non remboursable annulé à 99,2 %** | **10 461** | Personne ne verse un acompte non remboursable pour annuler dans 99 % des cas. Le profil est très particulier : 61 % de groupes, 96 % de clients portugais, 87 % au City Hotel, réservés 238 jours à l'avance (médiane, contre 65 pour la base), et 93 % sont des doublons. On dirait des **blocs de chambres** posés par des agences puis libérés, ou un type d'acompte mal codé. |
| 5 | **Groupe de 20 à 55 adultes sur une seule ligne** | **12** | Toutes au Resort, toutes annulées, toutes à prix nul : des options de groupe, pas des séjours. |
| 6 | **Date de départ ≠ arrivée + nuits** | **23** | Statut `Check-Out` daté du jour de l'arrivée pour des séjours de 3 à 10 nuits : départ anticipé, ou date jamais mise à jour. |

**La plus importante pour le modèle est la n° 4.** Le modèle v2 a appris « Non Refund ⇒ annulation ». Si ces lignes sont des blocs de chambres et non de vraies réservations, une vraie réservation non remboursable envoyée à l'API sera prédite comme annulée, alors que c'est justement le client le **moins** susceptible d'annuler. Seul le client peut dire ce que ces lignes représentent.

## Tâche 3 — Les décisions

Rappel : `train.py` ne charge que les colonnes utiles au modèle. `babies`, `agent`, `company`, `reservation_status` et `reservation_status_date` n'en font **pas** partie : une anomalie sur ces colonnes n'a aucune influence sur le modèle.

| Anomalie | Lignes | Décision (supprimer / corriger / imputer / garder et signaler) | Justification | Question au client ? |
|---|---|---|---|---|
| `children` manquant | 4 | imputer à 0 (règle d'exemple) | Quatre lignes, et 0 enfant est de loin le cas le plus fréquent. | Non |
| `country` manquant | 405 | garder (le pipeline impute déjà `Unknown`) | Le manque porte de l'information (13,8 % d'annulation contre 36,3 %). Le remplacer par le pays le plus fréquent (PRT) effacerait ce signal. | Non (à quel moment le pays est-il saisi ? utile, pas bloquant) |
| `agent` / `company` manquants | 10 676 / 74 216 | garder | « Manquant » veut dire « pas d'intermédiaire ». Colonnes non utilisées par le modèle. | Non |
| Doublons | 22 999 | **garder et signaler** | Sans identifiant client ni numéro de chambre, on ne distingue pas une erreur d'une réservation de groupe (une ligne par chambre). Les supprimer retirerait 29 % de la base, dont 93 % des `Non Refund`. | **Oui** |
| `adr` = 5 400 € | 1 | **supprimer** (règle : `adr` > 1 000 €) | Dix fois au-dessus de tous les autres prix. La corriger en 54 € serait une supposition ; une seule ligne, la supprimer ne coûte rien. | Non |
| `lead_time` = 737 jours | 1 | garder | Plausible : la borne de 730 est arbitraire. (Cette ligne part de toute façon avec la règle « séjour sans nuit ».) | Non |
| `babies` = 9 et 10 | 2 | garder et signaler | Faute de frappe probable, mais `babies` n'est pas une variable du modèle. | Non |
| Séjour sans aucune nuit | 571 | **supprimer** | Le modèle prédit l'annulation de séjours à l'hôtel : un séjour sans nuit n'en est pas un, et ses variables calculées n'ont pas de sens (prix nul, durée nulle). 0,7 % de la base. | **Oui** (location à la journée ?) |
| Réservation sans personne (0 adulte, 0 enfant, 0 bébé) | 113 | **supprimer** | Une réservation vide n'existe pas : c'est une saisie ratée. | Non |
| Réservation avec seulement des enfants ou des bébés | 160 | garder et signaler | Probablement une vraie réservation dont les adultes n'ont pas été saisis (172 des 273 lignes sans adulte viennent d'agences en ligne). Le reste de la ligne est cohérent. | **Oui** |
| Plus de 10 adultes sur une ligne | 12 | **supprimer** (règle : `adults` > 10) | Des options de groupe (toutes annulées, toutes à prix nul), pas des réservations. Elles apprendraient au modèle « gros groupe ⇒ annulation » sur 12 exemples. | Oui (liée à la question sur les groupes) |
| Prix nul d'un séjour offert (`Complementary`) | 479 | garder | Un séjour offert a un prix nul : c'est normal. | Non |
| Prix nul sans explication (nuits > 0, non offert) | 471 | garder et signaler | Peut-être un tarif facturé ailleurs (contrat, note du groupe). Les supprimer retirerait de vrais groupes. | **Oui** |
| `Non Refund` annulé à 99,2 % | 10 461 | **garder et signaler** | 13 % de la base et une variable très forte du modèle : trop lourd pour être tranché sans le client. Supprimer ou recoder ces lignes changerait profondément le modèle. | **Oui, la plus importante** |
| Date de départ ≠ arrivée + nuits | 23 | garder et signaler | `reservation_status_date` n'est pas utilisée par le modèle (elle trahirait la réponse). | Non |

**Bilan** : une imputation et quatre règles de suppression, qui seront les règles de `nettoyage.py` (tâche 4). Tout le reste est gardé ; les cas douteux deviennent des questions au client.

In [ ]:
# Effet cumulé des règles de suppression retenues
a_supprimer = {
    "adr > 1 000 €": df["adr"] > 1000,
    "séjour sans nuit": zero_nuit,
    "réservation vide": personnes == 0,
    "plus de 10 adultes": df["adults"] > 10,
}
supprimees = pd.concat(a_supprimer, axis=1).any(axis=1)
print(pd.Series({regle: masque.sum() for regle, masque in a_supprimer.items()}).to_string())
print(f"Total sans double compte : {supprimees.sum()} lignes, soit {supprimees.mean():.1%} de la base")
print(f"Taux d'annulation des lignes supprimées : {df.loc[supprimees, 'is_canceled'].mean():.1%}"
      f" | des lignes gardées : {df.loc[~supprimees, 'is_canceled'].mean():.1%}")

### ✏️ Questions au client (par ordre d'importance)

1. **Acomptes non remboursables.** 99 % des réservations `Non Refund` sont annulées, presque toutes des réservations de groupe ou d'agence, très en avance, en lignes identiques. S'agit-il de vrais acomptes encaissés, ou de blocs de chambres posés par des agences puis libérés ? Le type d'acompte est-il bien enregistré ?
2. **Lignes identiques.** Une ligne correspond-elle à une chambre ? Les lignes identiques d'un même groupe sont-elles des chambres distinctes ? Pouvez-vous nous fournir un numéro de dossier ou de groupe ?
3. **Prix à 0 €.** 471 séjours non offerts ont un prix nul, surtout des groupes et des agences : le tarif est-il facturé ailleurs (contrat, note globale) ?
4. **Séjours sans nuit.** 571 réservations n'ont aucune nuit : louez-vous des chambres à la journée ? Faut-il les prédire ?
5. **Réservations sans adulte.** 160 réservations n'ont que des enfants, surtout via des agences en ligne : le nombre d'adultes est-il bien transmis par les agences ?

## Tâche 6 — Comparer v2.0.0 et v2.1.0

À faire **après** avoir branché le nettoyage et produit la v2.1.0 avec `python train.py`.
La fonction de comparaison est fournie : c'est le **choix du jeu de test** qui t'appartient.

In [ ]:
import joblib
from sklearn.metrics import f1_score, precision_score, recall_score

import train as T
from features import INPUT_FEATURES


def comparer(modeles: dict, X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    lignes = []
    for nom, modele in modeles.items():
        pred = modele.predict(X)
        lignes.append({"modèle": nom, "f1": f1_score(y, pred), "rappel": recall_score(y, pred),
                       "précision": precision_score(y, pred), "n": len(y)})
    return pd.DataFrame(lignes).set_index("modèle").round(3)


modeles = {
    "v2.0.0 (données brutes)": joblib.load(ROOT / "model" / "cancellation_model_v2.0.0.joblib"),
    # "v2.1.0 (données nettoyées)": joblib.load(ROOT / "model" / "cancellation_model_v2.1.0.joblib"),
}

brut = T.load_data(engine)
_, test_brut = T.split_train_test(brut)
comparer(modeles, test_brut[INPUT_FEATURES], test_brut[T.TARGET])

In [ ]:
# TODO : construire le test nettoyé, et comparer les deux modèles dessus aussi

### ✏️ Interprétation

1. Le score monte, baisse ou reste stable : pourquoi ? Une baisse est-elle forcément une mauvaise nouvelle ?
2. Quelles réservations ont été exclues, et que fera le modèle s'il en reçoit de semblables en production ?
3. Faut-il appliquer le même nettoyage aux réservations que recevra l'API ?
4. Lequel des deux jeux de test, brut ou nettoyé, ressemble le plus à la production ? Lequel retiens-tu pour décider ?

*…*